# PCA-reconstructed spline intervention before layer 2

This notebook compares four interventions at `blocks.2.hook_resid_pre`:

1. **Clean:** use the model's original activation.
2. **Oracle PCA reconstruction:** project the original activation into the global 3D PCA subspace and reconstruct it. This isolates the cost of the 3D bottleneck.
3. **Spline → PCA reconstruction:** predict the three PCA coordinates from phase, line length, boundary residual, and token position, then map them back to `d_model` with `PCA.inverse_transform`.
4. **Full-space spline:** predict all `d_model` activation coordinates directly.

All conditions use identical held-out examples. Metrics include overall accuracy, newline precision, newline recall, and newline prediction frequency.

In [7]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import SplineTransformer

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
N_TRAIN_EXAMPLES = 2_000
N_EVAL_EXAMPLES = 1_000
COLLECTION_BATCH_SIZE = 128
TRAIN_SEED = 24_001
EVAL_SEED = 24_002
N_SPLINE_KNOTS = 12
RIDGE_ALPHA = 1.0


In [8]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

n_layers = cfg["model"]["n_layers"]
if n_layers < 3:
    raise ValueError("This experiment requires layer 2.")
PATCH_SITE = "blocks.2.hook_resid_pre"
seq_len = cfg["task"]["seq_len"]
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Intervention site: {PATCH_SITE}")


Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Intervention site: blocks.2.hook_resid_pre


## Collect training activations

In [9]:
def character_counts_for_tokens(token_rows):
    all_counts = []
    for row in token_rows.detach().cpu().tolist():
        running_count = 0
        row_counts = []
        for token_id in row:
            if token_id == vocab.newline_id:
                running_count = 0
            else:
                running_count += project_data.extract_character_count(vocab.decode_token(token_id))
            row_counts.append(running_count)
        all_counts.append(row_counts)
    return torch.tensor(all_counts, dtype=torch.float32)

@torch.no_grad()
def collect_training_activations(n_examples, seed):
    activation_chunks, count_chunks, width_chunks = [], [], []
    collected = 0
    batch_number = 0
    while collected < n_examples:
        size = min(COLLECTION_BATCH_SIZE, n_examples - collected)
        batch = project_data.make_batch(
            batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device,
            seed=seed + batch_number * COLLECTION_BATCH_SIZE,
        )
        _, cache = model.run_with_cache(
            batch.tokens, prepend_bos=False,
            names_filter=lambda name: name == PATCH_SITE,
        )
        activation_chunks.append(cache[PATCH_SITE].detach().cpu())
        count_chunks.append(character_counts_for_tokens(batch.tokens))
        width_chunks.append(batch.line_lengths.detach().cpu().float())
        collected += size
        batch_number += 1
    return (
        torch.cat(activation_chunks, dim=0)[:n_examples].numpy(),
        torch.cat(count_chunks, dim=0)[:n_examples].numpy(),
        torch.cat(width_chunks, dim=0)[:n_examples].numpy(),
    )

X_train, counts_train, widths_train = collect_training_activations(N_TRAIN_EXAMPLES, TRAIN_SEED)
d_model = X_train.shape[-1]
print("Training activations:", X_train.shape)


Training activations: (2000, 150, 128)


## Fit global PCA and trajectory models

In [10]:
pca = PCA(n_components=3, svd_solver="randomized", random_state=24_003)
pca.fit(X_train.reshape(-1, d_model))
Y_train = pca.transform(X_train.reshape(-1, d_model))

width_mean = widths_train.mean()
width_std = max(widths_train.std(), 1.0)
phase_spline = SplineTransformer(n_knots=N_SPLINE_KNOTS, degree=3, include_bias=False)
phase_spline.fit(np.linspace(0.0, 1.5, 500)[:, None])
spline_feature_count = phase_spline.n_features_out_

def make_features(character_counts, line_lengths):
    n_examples, n_positions = character_counts.shape
    phase = np.clip(character_counts / line_lengths[:, None], 0.0, 1.5)
    line_length_centered = (line_lengths - width_mean) / width_std
    boundary_residual = (character_counts - line_lengths[:, None]) / max(width_mean, 1.0)
    token_position = np.broadcast_to(
        np.arange(n_positions, dtype=np.float32) / max(n_positions - 1, 1),
        (n_examples, n_positions),
    )
    spline_features = phase_spline.transform(phase.reshape(-1, 1))
    line_feature = np.repeat(line_length_centered, n_positions)[:, None]
    return np.concatenate([
        spline_features,
        spline_features * line_feature,
        boundary_residual.reshape(-1, 1),
        token_position.reshape(-1, 1),
    ], axis=1)

F_train = make_features(counts_train, widths_train)
pca_spline_model = Ridge(alpha=RIDGE_ALPHA)
pca_spline_model.fit(F_train, Y_train)

full_spline_model = Ridge(alpha=RIDGE_ALPHA)
full_spline_model.fit(F_train, X_train.reshape(-1, d_model))

print(f"Global PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print(f"Spline features: {F_train.shape[1]}")
print(f"PCA-coordinate spline train R²: {pca_spline_model.score(F_train, Y_train):.4f}")
print(f"Full-space spline train R²: {full_spline_model.score(F_train, X_train.reshape(-1, d_model)):.4f}")


Global PCA variance explained: 88.4%
Spline features: 28
PCA-coordinate spline train R²: 0.8159
Full-space spline train R²: 0.7486


## Evaluate clean, PCA, and spline interventions

In [11]:
pca_mean = torch.as_tensor(pca.mean_, device=device, dtype=model.W_Q.dtype)
pca_components = torch.as_tensor(pca.components_, device=device, dtype=model.W_Q.dtype)

@torch.no_grad()
def evaluate_condition(condition):
    correct = total = true_positives = actual_newlines = predicted_newlines = 0
    for start in range(0, N_EVAL_EXAMPLES, COLLECTION_BATCH_SIZE):
        size = min(COLLECTION_BATCH_SIZE, N_EVAL_EXAMPLES - start)
        batch = project_data.make_batch(
            batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device,
            seed=EVAL_SEED + start,
        )
        hooks = []
        if condition != "clean":
            counts = character_counts_for_tokens(batch.tokens).numpy()
            widths = batch.line_lengths.detach().cpu().numpy().astype(np.float32)
            features = make_features(counts, widths)

            if condition == "oracle_pca":
                def replace_with_oracle_pca(activation, hook):
                    centered = activation - pca_mean.to(activation.dtype)
                    coords = centered @ pca_components.to(activation.dtype).T
                    return coords @ pca_components.to(activation.dtype) + pca_mean.to(activation.dtype)
                hook_fn = replace_with_oracle_pca
            elif condition == "pca_spline":
                coords = pca_spline_model.predict(features).reshape(size, seq_len, 3)
                replacement = torch.as_tensor(coords, device=device, dtype=model.W_Q.dtype)
                replacement = replacement @ pca_components + pca_mean
                def replace_with_pca_spline(activation, hook, replacement=replacement):
                    return replacement.to(device=activation.device, dtype=activation.dtype)
                hook_fn = replace_with_pca_spline
            elif condition == "full_spline":
                replacement = full_spline_model.predict(features).reshape(size, seq_len, d_model)
                replacement = torch.as_tensor(replacement, device=device, dtype=model.W_Q.dtype)
                def replace_with_full_spline(activation, hook, replacement=replacement):
                    return replacement.to(device=activation.device, dtype=activation.dtype)
                hook_fn = replace_with_full_spline
            else:
                raise ValueError(f"Unknown condition: {condition}")
            hooks = [(PATCH_SITE, hook_fn)]

        logits = model.run_with_hooks(batch.tokens, prepend_bos=False, fwd_hooks=hooks)
        predictions = logits.argmax(dim=-1)
        targets = batch.targets
        target_newline = targets == vocab.newline_id
        prediction_newline = predictions == vocab.newline_id
        correct += int((predictions == targets).sum().item())
        total += predictions.numel()
        true_positives += int((target_newline & prediction_newline).sum().item())
        actual_newlines += int(target_newline.sum().item())
        predicted_newlines += int(prediction_newline.sum().item())

    return {
        "overall_accuracy": correct / total,
        "newline_precision": true_positives / predicted_newlines if predicted_newlines else np.nan,
        "newline_recall": true_positives / actual_newlines if actual_newlines else np.nan,
        "newline_prediction_frequency": predicted_newlines / total,
        "actual_newlines": actual_newlines,
        "predicted_newlines": predicted_newlines,
    }

conditions = ["clean", "oracle_pca", "pca_spline", "full_spline"]
metrics = {condition: evaluate_condition(condition) for condition in conditions}
results = pd.DataFrame.from_dict(metrics, orient="index").reset_index().rename(columns={"index": "condition"})
display(results.style.format({
    "overall_accuracy": "{:.4f}",
    "newline_precision": "{:.4f}",
    "newline_recall": "{:.4f}",
    "newline_prediction_frequency": "{:.4f}",
}))


,condition,overall_accuracy,newline_precision,newline_recall,newline_prediction_frequency,actual_newlines,predicted_newlines
0,clean,0.0651,0.5224,0.9996,0.1031,8082,15466
1,oracle_pca,0.0645,0.4657,0.9939,0.1150,8082,17251
2,pca_spline,0.0648,0.4586,0.9976,0.1172,8082,17581
3,full_spline,0.0647,0.5054,0.9963,0.1062,8082,15932


## Changes relative to clean

In [13]:
metric_names = [
    "overall_accuracy", "newline_precision", "newline_recall",
    "newline_prediction_frequency",
]
change_table = results[["condition"] + metric_names].copy()
for metric in metric_names:
    clean_value = metrics["clean"][metric]
    change_table[f"{metric}_change"] = change_table[metric] - clean_value
display(change_table.style.format({metric: "{:+.4f}" for metric in metric_names}))


,condition,overall_accuracy,newline_precision,newline_recall,newline_prediction_frequency,overall_accuracy_change,newline_precision_change,newline_recall_change,newline_prediction_frequency_change
0,clean,+0.0651,+0.5224,+0.9996,+0.1031,0.000000,0.000000,0.000000,0.000000
1,oracle_pca,+0.0645,+0.4657,+0.9939,+0.1150,-0.000633,-0.056717,-0.005692,0.011900
2,pca_spline,+0.0648,+0.4586,+0.9976,+0.1172,-0.000327,-0.063752,-0.001980,0.014100
3,full_spline,+0.0647,+0.5054,+0.9963,+0.1062,-0.000480,-0.016974,-0.003341,0.003107
